# Monthly Aggregation v2 — `product_id x month` panel for Revenue & Quantity forecasting

**Why this notebook exists:** the previous EDA showed that the
`(product_id, client_id, month)` panel is structurally sound but too noisy at
that grain — 43% zero months, extreme skew (37 / 167), and a weak
`lag1` signal (corr ≈ 0.26 in log space). Collapsing to **`product_id x month`**
roughly halved the zero-rate and skew and **doubled** the lag1 signal
(0.26 → 0.57).

This notebook:

1. Builds a new **`product_month_panel`** at `(product_id, month)` grain,
   carrying forward everything useful from `client_id` as **aggregated
   features** (not as part of the grouping key).
2. Applies all the other fixes recommended previously: log targets, fixed
   `month_dt`, family/therapeutic-area fallback features, sparse-series
   handling, lag/rolling features, December-aware calendar features.
3. Runs a focused before/after EDA to confirm the new panel is actually more
   learnable — same metrics as before, side by side.

**Targets remain `revenue` and `quantity`** (now summed across all clients for
a product in a given month).

## 1. Load data & basic setup

In [4]:
import os
print(os.getcwd())

c:\Users\bochr\OneDrive\Documents\projet Medis\data\processed


In [5]:
import pandas as pd
import numpy as np

pd.set_option('display.float_format', '{:,.2f}'.format)

df = pd.read_csv('merged_sales.csv')

df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['month'] = df['invoice_date'].dt.to_period('M')

print(f"merged_sales: {df.shape[0]:,} rows")
print(f"Date range  : {df['invoice_date'].min().date()} to {df['invoice_date'].max().date()} "
      f"({df['month'].nunique()} months)")
print(f"Products    : {df['product_id'].nunique()}")
print(f"Clients     : {df['client_id'].nunique()}")

merged_sales: 290,542 rows
Date range  : 2023-01-02 to 2026-04-30 (40 months)
Products    : 641
Clients     : 170


## 2. Verify product-level attributes are 1:1 with `product_id`

Same check as the original aggregation notebook, but now this is the
**only** constancy check we need — `product_id` is the entire grouping key,
so anything that's 1:1 with `product_id` can be carried through with
`first()`; anything that varies *within* a product (because it was a
client-level attribute) must become a **derived/aggregated feature** instead
(handled in section 4).

In [6]:
product_attr_cols = ['product_name', 'pack_coefficient', 'therapeutic_area_clean', 'product_family_clean']
print("Product attributes constant within product_id:")
for c in product_attr_cols:
    n = df.groupby('product_id')[c].nunique()
    print(f"  {c:<24} -> {(n>1).sum()} products with >1 value (out of {len(n)})")

print()
print("Former client-level attributes that now VARY within product_id (as expected):")
for c in ['market_type', 'client_channel', 'region', 'client_type']:
    n = df.groupby('product_id')[c].nunique()
    print(f"  {c:<24} -> {(n>1).sum()} / {len(n)} products see >1 value")

Product attributes constant within product_id:
  product_name             -> 0 products with >1 value (out of 641)
  pack_coefficient         -> 0 products with >1 value (out of 641)
  therapeutic_area_clean   -> 0 products with >1 value (out of 641)
  product_family_clean     -> 0 products with >1 value (out of 641)

Former client-level attributes that now VARY within product_id (as expected):
  market_type              -> 217 / 641 products see >1 value
  client_channel           -> 149 / 641 products see >1 value
  region                   -> 206 / 641 products see >1 value
  client_type              -> 282 / 641 products see >1 value


**Result:** product attributes (`product_name`, `pack_coefficient`,
`therapeutic_area_clean`, `product_family_clean`) remain 1:1 with
`product_id`, so `first()` is still correct for those. As expected,
`market_type`/`region`/`client_channel`/`client_type` now vary within a
product (because most products sell to many different clients) — these are
turned into **share/diversity features** below instead of single categorical
values.

## 3. Core aggregation: targets, prices, discounts, static product attributes

Same logic as before, just at `(product_id, month)`:

- **Targets**: `revenue`, `quantity` → `sum`
- **Diagnostics**: `unit_price` → `min`/`max` (price dispersion); `n_invoices` = distinct invoices
- **Weighted price**: `unit_price_w = revenue / quantity` — weighted average price
  across all clients for that product-month
- **Static product attributes** via `first()`

In [7]:
panel_key = ['product_id', 'month']

g = df.groupby(panel_key, observed=True)

panel = g.agg(
    revenue=('revenue', 'sum'),
    quantity=('quantity', 'sum'),
    unit_price_min=('unit_price', 'min'),
    unit_price_max=('unit_price', 'max'),
    n_invoices=('invoice_id', 'nunique'),
    product_name=('product_name', 'first'),
    pack_coefficient=('pack_coefficient', 'first'),
    therapeutic_area_clean=('therapeutic_area_clean', 'first'),
    product_family_clean=('product_family_clean', 'first'),
).reset_index()

panel['unit_price_w'] = panel['revenue'] / panel['quantity'].replace(0, np.nan)

print(f"panel shape (pre-fill): {panel.shape}")
panel.head()

panel shape (pre-fill): (10570, 12)


,product_id,month,revenue,quantity,unit_price_min,unit_price_max,n_invoices,product_name,pack_coefficient,therapeutic_area_clean,product_family_clean,unit_price_w
0,ABI250T120,2023-01,"10,874.86",8,"1,387.10","1,387.10",1,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,ABIRATERONE,"1,359.36"
1,ABI250T120,2023-02,"58,452.39",43,"1,387.10","1,387.10",5,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,ABIRATERONE,"1,359.36"
2,ABI250T120,2023-03,"48,936.89",36,"1,387.10","1,387.10",5,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,ABIRATERONE,"1,359.36"
3,ABI250T120,2023-04,"13,593.58",10,"1,387.10","1,387.10",1,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,ABIRATERONE,"1,359.36"
4,ABI250T120,2023-08,"107,389.28",79,"1,387.10","1,387.10",3,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,ABIRATERONE,"1,359.36"


## 4. Client-mix features (replacing `client_id` as a grouping key)

`client_id` and its attributes (`region`, `market_type`, `client_channel`,
`client_type`, `commission_rate`) used to fully determine the row's context.
At `product_id x month` they instead describe **who bought this product this
month** — that's a useful signal (e.g. a sudden hospital tender, or a new
export market opening up), so we summarize it as features per
`(product_id, month)`:

- `n_active_clients` — how many distinct clients bought this product this month
- `export_rev_share` — share of this month's revenue from `market_type == 'EXPORT'`
- `hospital_rev_share` — share of revenue from `client_channel == 'HOPITAUX'`
- `n_regions` — how many distinct regions bought this product this month
- `commission_rate_w` — revenue-weighted average commission rate

These are exactly the kind of "client-level info, but as covariates instead
of grouping keys" that the previous assessment recommended.

In [8]:
def rev_share(group, col, value):
    tot = group['revenue'].sum()
    if tot == 0:
        return 0.0
    return group.loc[group[col] == value, 'revenue'].sum() / tot

client_mix = g.apply(lambda gr: pd.Series({
    'n_active_clients'  : gr['client_id'].nunique(),
    'export_rev_share'  : rev_share(gr, 'market_type', 'EXPORT'),
    'hospital_rev_share': rev_share(gr, 'client_channel', 'HOPITAUX'),
    'n_regions'         : gr['region'].nunique(),
    'commission_rate_w' : np.average(gr['commission_rate'], weights=gr['revenue'].replace(0, 1e-9)),
}), include_groups=False).reset_index()

panel = panel.merge(client_mix, on=panel_key, how='left')

panel[['n_active_clients','export_rev_share','hospital_rev_share','n_regions','commission_rate_w']].describe()

,n_active_clients,export_rev_share,hospital_rev_share,n_regions,commission_rate_w
count,"10,570.00","10,570.00","10,570.00","10,570.00","10,570.00"
mean,21.86,0.19,0.08,1.13,0.03
std,17.56,0.36,0.26,0.38,0.09
min,1.00,0.00,0.00,1.00,0.00
25%,3.00,0.00,0.00,1.00,0.00
50%,20.00,0.00,0.00,1.00,0.00
75%,38.00,0.15,0.00,1.00,0.00
max,64.00,1.00,1.00,4.00,0.40


**What this captures that pure `(product, client, month)` lost:** breadth
of demand (`n_active_clients`, `n_regions`) and channel/market mix
(`export_rev_share`, `hospital_rev_share`) for each product each month — signals
that disappear if you only look at one client at a time, but matter a lot for
predicting a product's *total* monthly revenue (e.g. a product suddenly sold
to 10 new clients is a strong leading indicator of higher volume).

## 5. Calendar features

Same as before — `year`, `month_num`, `quarter` — built directly from the
`month` period (so no ambiguous date-string parsing).

In [9]:
panel['month_dt']  = panel['month'].dt.to_timestamp()
panel['year']      = panel['month'].dt.year
panel['month_num'] = panel['month'].dt.month
panel['quarter']   = panel['month'].dt.quarter

# Explicit December flag, since the previous EDA found a consistent
# year-over-year December dip in aggregate revenue.
panel['is_december'] = (panel['month_num'] == 12).astype(int)

panel = panel.sort_values(['product_id', 'month_dt']).reset_index(drop=True)
panel[['product_id','month','month_dt','year','month_num','quarter','is_december']].head()

,product_id,month,month_dt,year,month_num,quarter,is_december
0,ABI250T120,2023-01,2023-01-01,2023,1,1,0
1,ABI250T120,2023-02,2023-02-01,2023,2,1,0
2,ABI250T120,2023-03,2023-03-01,2023,3,1,0
3,ABI250T120,2023-04,2023-04-01,2023,4,2,0
4,ABI250T120,2023-08,2023-08-01,2023,8,3,0


**Fix applied (recommendation #4 from the previous review):** `month_dt`
is built directly from the `month` Period column (`to_timestamp()`), which is
unambiguous — no risk of day/month transposition that the old CSV's
`M/D/YYYY` strings carried. The new `is_december` flag makes the
year-over-year December dip found in section 8 of the previous EDA an
explicit, easy-to-use feature rather than something the model has to infer
from `month_num == 12` interacting with revenue scale.

## 6. Sparse-series flag (`is_sparse`)

Same definition as before, but now at the `product_id` series level: a
product with fewer than 3 active months total has too little history for
lag/rolling features.

In [10]:
SPARSE_THRESHOLD = 3

series_months = panel.groupby('product_id')['month'].nunique()
sparse_products = series_months[series_months < SPARSE_THRESHOLD].index
panel['is_sparse'] = panel['product_id'].isin(sparse_products)

print(f"Sparse threshold        : < {SPARSE_THRESHOLD} active months")
print(f"Sparse products         : {len(sparse_products):,} / {series_months.shape[0]:,} "
      f"({len(sparse_products)/series_months.shape[0]:.1%})")
print(f"Rows flagged is_sparse   : {panel['is_sparse'].sum():,} / {len(panel):,} "
      f"({panel['is_sparse'].mean():.1%})")
print(f"Revenue share in sparse  : {panel.loc[panel.is_sparse,'revenue'].sum()/panel['revenue'].sum():.2%}")

Sparse threshold        : < 3 active months
Sparse products         : 155 / 641 (24.2%)
Rows flagged is_sparse   : 226 / 10,570 (2.1%)
Revenue share in sparse  : 1.70%


## 7. Fill to a regular monthly grid

Each product gets one row per calendar month between its first and last
active month, with `revenue = 0`, `quantity = 0` etc. for silent months —
same approach as before, just one series per `product_id` instead of per
`(product_id, client_id)`.

In [11]:
def fill_series(grp):
    full_range = pd.period_range(grp['month'].min(), grp['month'].max(), freq='M')
    grp = grp.set_index('month').reindex(full_range)
    grp.index.name = 'month'
    return grp

filled = (panel.groupby('product_id', group_keys=True)
                .apply(fill_series, include_groups=False)
                .reset_index())

zero_fill_cols = ['revenue', 'quantity', 'n_invoices',
                  'n_active_clients', 'n_regions']
filled[zero_fill_cols] = filled[zero_fill_cols].fillna(0)

static_cols = ['product_name', 'pack_coefficient', 'therapeutic_area_clean',
                'product_family_clean', 'is_sparse']
filled[static_cols] = (filled.groupby('product_id', group_keys=False)[static_cols]
                              .apply(lambda gg: gg.ffill().bfill()))

filled['month_dt']  = filled['month'].dt.to_timestamp()
filled['year']      = filled['month'].dt.year
filled['month_num'] = filled['month'].dt.month
filled['quarter']   = filled['month'].dt.quarter
filled['is_december'] = (filled['month_num'] == 12).astype(int)

# price/discount/mix columns: no observation in a silent month -> carry forward last known
price_mix_cols = ['unit_price_w','unit_price_min','unit_price_max',
                   'export_rev_share','hospital_rev_share','commission_rate_w']
filled[price_mix_cols] = (filled.groupby('product_id', group_keys=False)[price_mix_cols]
                                  .apply(lambda gg: gg.ffill().bfill()))

print(f"Rows before filling: {len(panel):,}")
print(f"Rows after filling : {len(filled):,}")
filled.head(8)

Rows before filling: 10,570
Rows after filling : 15,413


,product_id,month,revenue,quantity,unit_price_min,unit_price_max,n_invoices,product_name,pack_coefficient,therapeutic_area_clean,...,export_rev_share,hospital_rev_share,n_regions,commission_rate_w,month_dt,year,month_num,quarter,is_december,is_sparse
0,ABI250T120,2023-01,"10,874.86",8.00,"1,387.10","1,387.10",1.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,1.00,0.00,2023-01-01,2023,1,1,0,False
1,ABI250T120,2023-02,"58,452.39",43.00,"1,387.10","1,387.10",5.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,1.00,0.00,2023-02-01,2023,2,1,0,False
2,ABI250T120,2023-03,"48,936.89",36.00,"1,387.10","1,387.10",5.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,1.00,0.00,2023-03-01,2023,3,1,0,False
3,ABI250T120,2023-04,"13,593.58",10.00,"1,387.10","1,387.10",1.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,1.00,0.00,2023-04-01,2023,4,2,0,False
4,ABI250T120,2023-05,0.00,0.00,"1,387.10","1,387.10",0.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,0.00,0.00,2023-05-01,2023,5,2,0,False
5,ABI250T120,2023-06,0.00,0.00,"1,387.10","1,387.10",0.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,0.00,0.00,2023-06-01,2023,6,2,0,False
6,ABI250T120,2023-07,0.00,0.00,"1,387.10","1,387.10",0.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,0.00,0.00,2023-07-01,2023,7,3,0,False
7,ABI250T120,2023-08,"107,389.28",79.00,"1,387.10","1,387.10",3.00,ABIRATERONE NEAPOLIS 250 MG,120.00,Oncology & Immunology,...,0.00,0.00,1.00,0.00,2023-08-01,2023,8,3,0,False


## 8. Lag & rolling features

Same as before: `lag1`/`lag2`/`lag3` and a 3-month rolling mean (computed on
`shift(1)`, so no leakage) for both `revenue` and `quantity`.

In [12]:
for col in ['revenue', 'quantity']:
    for lag in [1, 2, 3]:
        filled[f'{col}_lag{lag}'] = filled.groupby('product_id')[col].shift(lag)
    filled[f'{col}_roll3_mean'] = (filled.groupby('product_id')[col]
                                          .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean()))

filled[['product_id','month','revenue','revenue_lag1','revenue_lag2','revenue_lag3','revenue_roll3_mean']].head(8)

,product_id,month,revenue,revenue_lag1,revenue_lag2,revenue_lag3,revenue_roll3_mean
0,ABI250T120,2023-01,"10,874.86",NaN,NaN,NaN,NaN
1,ABI250T120,2023-02,"58,452.39","10,874.86",NaN,NaN,"10,874.86"
2,ABI250T120,2023-03,"48,936.89","58,452.39","10,874.86",NaN,"34,663.63"
3,ABI250T120,2023-04,"13,593.58","48,936.89","58,452.39","10,874.86","39,421.38"
4,ABI250T120,2023-05,0.00,"13,593.58","48,936.89","58,452.39","40,327.62"
5,ABI250T120,2023-06,0.00,0.00,"13,593.58","48,936.89","20,843.49"
6,ABI250T120,2023-07,0.00,0.00,0.00,"13,593.58","4,531.19"
7,ABI250T120,2023-08,"107,389.28",0.00,0.00,0.00,0.00


## 9. Log targets

Per recommendation, both targets get a `log1p` version for modeling, given
the extreme skew found previously.

In [13]:
filled['log_revenue'] = np.log1p(filled['revenue'])
filled['log_quantity'] = np.log1p(filled['quantity'])

filled[['revenue','log_revenue','quantity','log_quantity']].describe()

,revenue,log_revenue,quantity,log_quantity
count,"15,413.00","15,413.00","15,413.00","15,413.00"
mean,"67,392.13",7.06,"9,101.21",5.24
std,"179,259.10",5.00,"59,609.80",3.94
min,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00
50%,"11,568.96",9.36,700.00,6.55
75%,"61,908.49",11.03,"4,826.00",8.48
max,"3,643,131.27",15.11,"3,208,940.00",14.98


## 10. Save the new panel

In [14]:
print(f"Final product_month_panel shape: {filled.shape}")
print(f"Columns: {list(filled.columns)}")

filled.to_csv('product_month_panel.csv', index=False)
print("\nSaved: product_month_panel.csv")

Final product_month_panel shape: (15413, 33)
Columns: ['product_id', 'month', 'revenue', 'quantity', 'unit_price_min', 'unit_price_max', 'n_invoices', 'product_name', 'pack_coefficient', 'therapeutic_area_clean', 'product_family_clean', 'unit_price_w', 'n_active_clients', 'export_rev_share', 'hospital_rev_share', 'n_regions', 'commission_rate_w', 'month_dt', 'year', 'month_num', 'quarter', 'is_december', 'is_sparse', 'revenue_lag1', 'revenue_lag2', 'revenue_lag3', 'revenue_roll3_mean', 'quantity_lag1', 'quantity_lag2', 'quantity_lag3', 'quantity_roll3_mean', 'log_revenue', 'log_quantity']

Saved: product_month_panel.csv
